# Submit the Integration Compare Pipeline

Load the workshop-local five-step pipeline YAML, replace compute placeholders from `.env`, submit it, stream the parent run, and inspect child-job status.

**Source:** Adapted from this repository's `notebooks/00_submit_azureml_pipelines.ipynb` and registry-free `pipelines/integration-compare-pipeline.yaml`.

In [ ]:
from pathlib import Path
import os

import pandas as pd
from azure.ai.ml import MLClient, load_job
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / ".env.example").is_file() and (candidate / "pipelines").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")
load_dotenv(WORKSHOP_ROOT / ".env", override=True)

credential = AzureCliCredential(tenant_id=os.getenv("AZURE_TENANT_ID") or None)
ml_client = MLClient(credential, os.environ["AZURE_SUBSCRIPTION_ID"], os.environ["AZURE_RESOURCE_GROUP"], os.environ["AZUREML_WORKSPACE_NAME"])
COMPUTE_NAME = os.environ["AZUREML_COMPUTE_NAME"]
OUTPUT_DATASTORE = os.getenv("AZUREML_OUTPUT_DATASTORE", "workspaceblobstore")
RUN_JOB = os.getenv("RUN_INTEGRATION_PIPELINE", "false").lower() in {"1", "true", "yes"}
pipeline_path = WORKSHOP_ROOT / "pipelines/integration-compare-pipeline.yaml"

In [ ]:
job = load_job(pipeline_path)
job.settings.default_compute = f"azureml:{COMPUTE_NAME}"
job.settings.default_datastore = f"azureml:{OUTPUT_DATASTORE}"
job.inputs["automl_compute"] = COMPUTE_NAME
job.display_name = "Workshop taxi integration compare"
job.tags = {"workshop": "azureml-h2o", "operation": "integration-compare"}

print(f"Loaded: {type(job).__name__}")
print(f"Compute: {COMPUTE_NAME}")
print(f"Datastore: {OUTPUT_DATASTORE}")

if RUN_JOB:
    submitted_job = ml_client.jobs.create_or_update(job)
    print(f"Submitted: {submitted_job.name}")
    ml_client.jobs.stream(submitted_job.name)
    final_job = ml_client.jobs.get(submitted_job.name)
    child_jobs = list(ml_client.jobs.list(parent_job_name=submitted_job.name))
    child_summary = pd.DataFrame(
        [{"name": child.name, "display_name": child.display_name, "status": child.status} for child in child_jobs]
    )
    display(child_summary)
    if final_job.status != "Completed":
        raise RuntimeError(f"Pipeline ended with status {final_job.status}")
    if any(child.status != "Completed" for child in child_jobs):
        raise RuntimeError("At least one child job did not complete")
    print({name: output.path for name, output in final_job.outputs.items()})
else:
    print("Submission disabled. Set RUN_INTEGRATION_PIPELINE=true in workshop/.env.")

## Expected Result

The parent pipeline and all five active child jobs complete, producing trained-model, prediction, and comparison outputs.

Next: `../03_h2o_reference/01_create_reference_binary_model.ipynb`.